
## 1. Naive Bayes with Laplace Smoothing




In [ ]:

from collections import Counter

data = [
    ("Weak", "High", "Yes"),
    ("Strong", "High", "Yes"),
    ("Weak", "High", "No"),
    ("Weak", "Low", "Yes"),
    ("Strong", "Low", "No")
]

# Class priors
yes_count = sum(row[2] == "Yes" for row in data)
no_count = sum(row[2] == "No" for row in data)
total = len(data)

P_yes = yes_count / total
P_no = no_count / total

print("P(Yes) =", P_yes)
print("P(No)  =", P_no)


P(Yes) = 0.6
P(No)  = 0.4



### Laplace-smoothed conditional probabilities



In [ ]:

# Naive Bayes prediction for every possible Wind/Humidity combination

def naive_bayes(wind, humidity):
    p_yes = (
        P_yes
        * ((3 + 1) / (yes_count + 2) if wind == "Weak" else (1 + 1) / (yes_count + 2))
        * ((2 + 1) / (yes_count + 2) if humidity == "High" else (1 + 1) / (yes_count + 2))
    )

    p_no = (
        P_no
        * 0.5
        * 0.5
    )

    total_prob = p_yes + p_no
    normalized_yes = p_yes / total_prob
    normalized_no = p_no / total_prob

    prediction = "Yes" if normalized_yes > normalized_no else "No"
    return p_yes, p_no, normalized_yes, normalized_no, prediction

for wind in ["Weak", "Strong"]:
    for humidity in ["High", "Low"]:
        print(wind, humidity, naive_bayes(wind, humidity))


Weak High (0.288, 0.1, 0.7422680412371133, 0.2577319587628866, 'Yes')
Weak Low (0.192, 0.1, 0.6575342465753424, 0.3424657534246575, 'Yes')
Strong High (0.144, 0.1, 0.5901639344262295, 0.4098360655737705, 'Yes')
Strong Low (0.096, 0.1, 0.4897959183673469, 0.5102040816326531, 'No')



### Naive Bayes example: Weak Wind + Low Humidity





## 2. Entropy and Information Gain



In [ ]:

import math

def entropy(labels):
    counts = Counter(labels)
    n = len(labels)
    return -sum((count/n) * math.log2(count/n) for count in counts.values())

labels = [row[2] for row in data]
base_entropy = entropy(labels)

print("Dataset entropy =", base_entropy)

for feature_index, feature_name in [(0, "Wind"), (1, "Humidity")]:
    groups = {}
    for row in data:
        groups.setdefault(row[feature_index], []).append(row[2])

    weighted_entropy = sum(
        (len(group_labels) / len(labels)) * entropy(group_labels)
        for group_labels in groups.values()
    )

    information_gain = base_entropy - weighted_entropy

    print("\nFeature:", feature_name)
    for value, group_labels in groups.items():
        print(value, "->", group_labels, "Entropy =", entropy(group_labels))
    print("Weighted entropy =", weighted_entropy)
    print("Information gain =", information_gain)


Dataset entropy = 0.9709505944546686

Feature: Wind
Weak -> ['Yes', 'No', 'Yes'] Entropy = 0.9182958340544896
Strong -> ['Yes', 'No'] Entropy = 1.0
Weighted entropy = 0.9509775004326937
Information gain = 0.01997309402197489

Feature: Humidity
High -> ['Yes', 'Yes', 'No'] Entropy = 0.9182958340544896
Low -> ['Yes', 'No'] Entropy = 1.0
Weighted entropy = 0.9509775004326937
Information gain = 0.01997309402197489



### Information Gain result




## 3. Logistic Regression



In [ ]:

import numpy as np
from sklearn.linear_model import LogisticRegression

# Weak=0, Strong=1
# Low=0, High=1
X = np.array([
    [0, 1],  # Weak, High -> Yes
    [1, 1],  # Strong, High -> Yes
    [0, 1],  # Weak, High -> No
    [0, 0],  # Weak, Low  -> Yes
    [1, 0]   # Strong, Low -> No
])

y = np.array([1, 1, 0, 1, 0])

model = LogisticRegression()
model.fit(X, y)

print("Intercept:", model.intercept_[0])
print("Coefficients:", model.coef_[0])

probabilities = model.predict_proba(X)[:, 1]
predictions = model.predict(X)

for i, (p, pred) in enumerate(zip(probabilities, predictions), start=1):
    print(f"Example {i}: P(Rain=Yes) = {p:.3f}, prediction = {pred}")


Intercept: 0.3767589446713993
Coefficients: [-0.14963018  0.14963055]
Example 1: P(Rain=Yes) = 0.629, prediction = 1
Example 2: P(Rain=Yes) = 0.593, prediction = 1
Example 3: P(Rain=Yes) = 0.629, prediction = 1
Example 4: P(Rain=Yes) = 0.593, prediction = 1
Example 5: P(Rain=Yes) = 0.557, prediction = 1



### Important observation about the dataset

The pair **Wind = Weak, Humidity = High** appears twice with opposite labels:

- Weak + High → Yes
- Weak + High → No

Therefore no classifier can perfectly determine the label from only these two features for that combination. This is why the algorithms may produce probabilities rather than a perfect classification.

The logistic regression result also depends on the training settings because the assignment does not specify them.
